# Systematic Multi-Asset Alpha Engine

> **Research question:** Does integrating ML return forecasts into a Black-Litterman framework improve risk-adjusted performance vs. equal-weight and standard Markowitz?

---

## Architecture

```
Market Data
    │
    ▼
Feature Engineering  ──[WF-1: no look-ahead]──►  18 factors / asset
    │
    ▼
ML Ensemble (Stacked)  ──[ML-1: two-stage OOF]──►  Return forecasts + IC
    │
    ▼
GJR-GARCH  ──►  Vol forecasts + regime (high / low)
    │
    ▼
Black-Litterman  ──[BL-1: τ=1/T]──►  Posterior expected returns
    │
    ├─── low-vol regime  ──►  Mean-Variance (CVXPY QP)
    └─── high-vol regime ──►  Min-CVaR (CVXPY LP)
    │
    ▼
Backtest  ──►  P&L, Sharpe, Drawdown
    │
    ▼
Statistical Validation  ──►  Block-Bootstrap · DSR · IC t-test
```

**Universe:** 15 ETFs across equities, sectors, fixed income, and commodities  
**Period:** 2020-01-01 – 2024-12-31  
**Rebalance:** Monthly  
**Training window:** 504 trading days (~2 years)

## 1 · Setup

In [ ]:
import sys
import warnings
import logging

sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="arch")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import run_walk_forward
from src.config import Config, BacktestConfig, MLConfig, OptimizerConfig
from src.visualization import plot_performance, plot_weights_and_risk

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
plt.style.use("seaborn-v0_8-darkgrid")
np.random.seed(42)

print("Environment ready.")

## 2 · Configuration

All parameters are centralised in `src/config.py`. Override individual fields here for experiments.

In [ ]:
cfg = Config(
    backtest=BacktestConfig(
        start_date="2020-01-01",
        end_date="2024-12-31",
        initial_capital=1_000_000,
        train_window=504,
        rebalance_freq="ME",
        run_ablation=True,
    ),
    ml=MLConfig(
        horizon=21,
        n_cv_splits=5,
    ),
    optimizer=OptimizerConfig(
        risk_aversion=2.5,
        risk_free_rate=0.04,
        max_weight=0.30,
        view_confidence=0.65,
    ),
)

print(f"Universe : {cfg.universe}")
print(f"Period   : {cfg.backtest.start_date} → {cfg.backtest.end_date}")
print(f"λ (risk aversion) : {cfg.optimizer.risk_aversion}")
print(f"Max weight        : {cfg.optimizer.max_weight:.0%}")

## 3 · Run the Walk-Forward Pipeline

This cell executes the complete pipeline:

1. Downloads 5 years of adjusted close prices for 15 ETFs
2. Runs 60 monthly rebalances, each with a fresh ML training window
3. Produces strategy and ablation (BL-only) portfolios
4. Reports statistical significance

> **Runtime:** ~15–30 minutes (GJR-GARCH fitting dominates). Set `run_ablation=False` to halve runtime.

In [ ]:
results = run_walk_forward(cfg)

## 4 · Performance Analysis

In [ ]:
bt = results["backtester"]
ph = bt.portfolio_history

print("\n" + "="*55)
print("  PERFORMANCE SUMMARY")
print("="*55)
for k, v in results["metrics"].items():
    print(f"  {k:<35} {v}")

if results["ablation_metrics"]:
    print("\n" + "="*55)
    print("  ABLATION — BL-only (no ML views)")
    print("="*55)
    for k, v in results["ablation_metrics"].items():
        print(f"  {k:<35} {v}")

In [ ]:
plot_performance(
    ph,
    benchmark=results["spy_returns"],
    title="Walk-Forward Alpha Engine (2020–2024)",
)

## 5 · Portfolio Allocation & Risk Decomposition

In [ ]:
tickers = results["common_tickers"]
wh = results["weights_history"]

plot_weights_and_risk(
    wh[tickers],
    results["prices"][tickers].pct_change().dropna(),
    wh[tickers].iloc[-1],
)

## 6 · ML Signal Quality

The Information Coefficient (IC) measures Spearman rank correlation between predicted and realised forward returns.  
An IC > 0 is a prerequisite for forwarding a view to the Black-Litterman model.

In [ ]:
ml_metrics = results["ml_metrics"]
ic_history = results["ic_history"]

if ml_metrics:
    ic_df = pd.DataFrame(ml_metrics).T[["IC", "r2", "calibration_slope"]]
    ic_df.columns = ["IC (OOF)", "R² (OOF)", "Calibration Slope"]
    display(ic_df.sort_values("IC (OOF)", ascending=False).round(4))

# IC evolution over time
ic_ts = pd.DataFrame(
    {t: v for t, v in ic_history.items() if len(v) >= 5}
)
if not ic_ts.empty:
    fig, ax = plt.subplots(figsize=(14, 4))
    ic_ts.plot(ax=ax, alpha=0.6, legend=True)
    ax.axhline(0, color="red", linestyle="--", linewidth=1)
    ax.set_title("IC Time Series by Asset", fontweight="bold")
    ax.set_ylabel("Information Coefficient")
    ax.legend(loc="upper right", ncol=5, fontsize=8)
    plt.tight_layout()
    plt.show()

## 7 · Conclusions

### What this engine demonstrates

| Technique | Purpose | Key design choice |
|---|---|---|
| Walk-forward CV | Prevent look-ahead bias | `cutoff_date` per rebalance |
| Stacked ensemble | Robust return forecasting | OOF meta-learner [ML-1] |
| GJR-GARCH | Leverage effect + regime | Skewed-t errors |
| Black-Litterman | Bayesian portfolio prior | τ = 1/T [BL-1] |
| Min-CVaR | Tail-risk aware allocation | High-vol regime switch |
| Block bootstrap | Autocorrelation-corrected p | block_size = 21d |
| Deflated Sharpe | Multiple-testing penalty | Harvey & Liu 2015 |

### Potential extensions

- **Alternative data**: Earnings sentiment NLP, options-implied vol surface features
- **Factor model covariance**: Barra-style statistical factor model vs. Ledoit-Wolf
- **Turnover constraints**: Add an L1 turnover penalty to the MV objective
- **Online learning**: Kalman-filter-based dynamic view updating
- **Transaction cost model**: Impact-adjusted costs using Almgren-Chriss